# Actividad 4. Métricas de calidad de resultados

## 1 Construcción de la muestra M

## 2 Construcción Train – Test

## 3 Selección de métricas para medir calidad de resultados

## 4 Entrenamiento de Modelos de Aprendizaje

## 5 Análisis de resultados

**Nombre:** Mario Salinas  
**Matrícula:** A01796938  
**Dataset:** Containers_Dataset.csv  

## Objetivo

El objetivo de esta actividad es identificar y aplicar métricas para medir la calidad de resultados derivados de modelos de aprendizaje automático supervisado y no supervisado, utilizando PySpark para el procesamiento de grandes volúmenes de datos.

Para esta actividad se utiliza el dataset `Containers_Dataset.csv`, relacionado con tráfico de red en ambientes de contenedores. A partir de este conjunto de datos se construirá una muestra representativa M, se generarán particiones de entrenamiento y prueba, se seleccionarán métricas de evaluación y se entrenarán modelos de aprendizaje automático para analizar la calidad de los resultados obtenidos.

## 1 Construcción de la muestra M

In [1]:
# Importación de librerías principales de PySpark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, isnull
from pyspark.sql.types import DoubleType

In [2]:
# Creación de la sesión de Spark

spark = SparkSession.builder \
    .appName("Actividad4_MetricasCalidadResultados") \
    .getOrCreate()

spark

### Inicialización del entorno de trabajo

En esta primera etapa se importaron las librerías principales de PySpark necesarias para el procesamiento distribuido de datos, la creación de nuevas variables y la manipulación de columnas dentro del DataFrame.

Posteriormente, se creó una sesión de Spark con el nombre `Actividad4_MetricasCalidadResultados`. Esta sesión será utilizada durante toda la actividad para cargar, transformar, particionar y analizar el conjunto de datos `Containers_Dataset.csv`.

El entorno quedó correctamente inicializado, utilizando Spark versión 4.1.1 en modo local con todos los núcleos disponibles (`local[*]`). Esto permite procesar el conjunto de datos de manera más eficiente que con herramientas tradicionales en memoria, lo cual es adecuado para trabajar con grandes volúmenes de datos.

In [4]:
# Carga del dataset original

df = spark.read.csv(
    r"C:\Users\masalin2\Downloads\Containers_Dataset.csv",
    header=True,
    inferSchema=True
)

# Visualización inicial de la estructura del dataset
print("Número de columnas:", len(df.columns))
print("Número de registros:", df.count())

df.printSchema()

Número de columnas: 87
Número de registros: 3231475
root
 |-- Flow ID: string (nullable = true)
 |-- Src IP: string (nullable = true)
 |-- Src Port: integer (nullable = true)
 |-- Dst IP: string (nullable = true)
 |-- Dst Port: integer (nullable = true)
 |-- Protocol: integer (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packet: integer (nullable = true)
 |-- Total Bwd packets: integer (nullable = true)
 |-- Total Length of Fwd Packet: double (nullable = true)
 |-- Total Length of Bwd Packet: double (nullable = true)
 |-- Fwd Packet Length Max: double (nullable = true)
 |-- Fwd Packet Length Min: double (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: double (nullable = true)
 |-- Bwd Packet Length Min: double (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std:

### Carga y exploración inicial de la población P

Una vez inicializada la sesión de Spark, se procedió a cargar el conjunto de datos original `Containers_Dataset.csv`, el cual representa la población P utilizada durante el desarrollo de la actividad.

La carga se realizó utilizando la función `spark.read.csv()`, habilitando las opciones `header=True` e `inferSchema=True` para que Spark identificara automáticamente los nombres de las columnas y los tipos de datos correspondientes.

Como resultado, se obtuvo una población compuesta por **3,231,475 registros** y **87 variables**, lo que confirma que se trata de un conjunto de datos de gran volumen adecuado para el análisis mediante tecnologías Big Data.

Posteriormente, se realizó una inspección de la estructura del dataset utilizando la función `printSchema()`. El análisis mostró la presencia de variables de distintos tipos, incluyendo atributos categóricos, direcciones IP, marcas de tiempo, puertos de red y una gran cantidad de métricas numéricas relacionadas con el comportamiento de los flujos de tráfico.

Entre las variables más relevantes destacan:

- **Protocol**, utilizada para identificar el protocolo de comunicación empleado por cada flujo.
- **Flow Duration**, que representa la duración total del flujo.
- **Flow Bytes/s** y **Flow Packets/s**, que describen la tasa de transferencia de datos.
- **Packet Length Mean** y **Packet Length Std**, que caracterizan el tamaño de los paquetes transmitidos.
- **Label**, utilizada como variable objetivo para identificar tráfico benigno o actividades potencialmente maliciosas.

La exploración inicial permitió validar que la información fue cargada correctamente y proporcionó una visión general de la estructura de la población antes de iniciar la construcción de la muestra representativa M.

# Sección 1: Construcción de la muestra M.

## Paso 1.1: crear traffic_type y protocol_type

In [6]:
from pyspark.sql.functions import col, when

# Construcción de variables de caracterización

df_caracterizado = df.withColumn(
    "traffic_type",
    when(col("Label") == 0, "Benign").otherwise("Malicious")
).withColumn(
    "protocol_type",
    when(col("Protocol") == 6, "TCP")
    .when(col("Protocol") == 17, "UDP")
    .otherwise("Other")
)

# Validación de las nuevas variables
df_caracterizado.select(
    "Label", "Protocol", "traffic_type", "protocol_type"
).show(10)

+-----+--------+------------+-------------+
|Label|Protocol|traffic_type|protocol_type|
+-----+--------+------------+-------------+
|    0|       0|      Benign|        Other|
|    0|       6|      Benign|          TCP|
|    0|       0|      Benign|        Other|
|    0|       0|      Benign|        Other|
|    0|       6|      Benign|          TCP|
|    0|       0|      Benign|        Other|
|    0|       6|      Benign|          TCP|
|    0|       6|      Benign|          TCP|
|    0|       6|      Benign|          TCP|
|    0|       6|      Benign|          TCP|
+-----+--------+------------+-------------+
only showing top 10 rows


### Definición de variables de caracterización

Con el objetivo de construir una muestra representativa M y generar particiones homogéneas de la población, se definieron dos variables de caracterización derivadas de atributos existentes en el conjunto de datos.

La primera variable fue denominada **traffic_type** y se construyó a partir de la columna **Label**. Para simplificar el análisis, los registros con valor 0 fueron clasificados como tráfico benigno (*Benign*), mientras que cualquier otro valor fue agrupado dentro de la categoría de tráfico malicioso (*Malicious*). Esta transformación permite diferenciar de manera general el comportamiento normal de la red respecto a posibles actividades de ataque.

La segunda variable fue denominada **protocol_type** y se derivó de la columna **Protocol**. Los registros con valor 6 fueron clasificados como tráfico TCP, aquellos con valor 17 como tráfico UDP y cualquier otro protocolo fue agrupado dentro de la categoría *Other*. Esta clasificación permite identificar diferencias de comportamiento asociadas al protocolo de transporte utilizado por cada flujo de red.

La creación de estas variables tiene como finalidad establecer criterios de particionamiento que permitan generar subconjuntos representativos de la población, minimizando la introducción de sesgos durante las etapas posteriores de muestreo, entrenamiento y evaluación de modelos.

La validación realizada sobre los primeros registros confirma que las transformaciones fueron aplicadas correctamente, observándose la correspondencia esperada entre los valores originales de las columnas **Label** y **Protocol** y las nuevas categorías generadas.

## Paso 1.2: revisar particiones Mi

In [8]:
# Identificación de particiones Mi por variables de caracterización

particiones_poblacion = df_caracterizado.groupBy(
    "traffic_type", "protocol_type"
).count().orderBy("traffic_type", "protocol_type")

particiones_poblacion.show(truncate=False)

+------------+-------------+-------+
|traffic_type|protocol_type|count  |
+------------+-------------+-------+
|Benign      |Other        |5235   |
|Benign      |TCP          |2905606|
|Benign      |UDP          |42450  |
|Malicious   |Other        |2524   |
|Malicious   |TCP          |254439 |
|Malicious   |UDP          |21221  |
+------------+-------------+-------+



### Identificación de las particiones Mi de la población

Una vez definidas las variables de caracterización **traffic_type** y **protocol_type**, se procedió a identificar las particiones que conforman la población P.

Cada partición Mi fue construida a partir de la combinación de ambas variables de caracterización, permitiendo agrupar registros con características similares respecto al tipo de tráfico y al protocolo utilizado. Como resultado, se identificaron seis particiones distintas:

| Partición | traffic_type | protocol_type |
|------------|------------|------------|
| M1 | Benign | Other |
| M2 | Benign | TCP |
| M3 | Benign | UDP |
| M4 | Malicious | Other |
| M5 | Malicious | TCP |
| M6 | Malicious | UDP |

El análisis de frecuencias muestra que la mayor parte de la población corresponde a tráfico benigno sobre protocolo TCP, con 2,905,606 registros, mientras que las demás particiones presentan tamaños considerablemente menores. Esta distribución refleja el comportamiento esperado en entornos reales de red, donde el tráfico legítimo suele representar la mayoría de las comunicaciones observadas.

Asimismo, se observa la presencia de tráfico malicioso tanto en protocolos TCP como UDP, así como un número reducido de registros clasificados dentro de la categoría Other. La existencia de estas particiones es relevante para el proceso de muestreo, ya que permite preservar la diversidad de comportamientos presentes en la población original.

Debido a que las particiones presentan tamaños muy diferentes entre sí, la utilización de un muestreo aleatorio simple podría introducir sesgos y reducir significativamente la representación de los grupos minoritarios. Por esta razón, en la siguiente etapa se aplicará un proceso de muestreo estratificado, garantizando que todas las particiones identificadas contribuyan proporcionalmente a la construcción de la muestra representativa M.

De esta manera, la población puede expresarse como:

\[
P = M_1 \cup M_2 \cup M_3 \cup M_4 \cup M_5 \cup M_6
\]

donde cada partición representa un subconjunto homogéneo definido por las variables de caracterización seleccionadas.

## Paso 1.3: construir muestra M

In [11]:
# Construcción de una llave de estrato para aplicar muestreo estratificado
from pyspark.sql.functions import concat_ws

from pyspark.sql.functions import (
    col,
    when,
    concat_ws,
    count
)

df_caracterizado = df_caracterizado.withColumn(
    "stratum",
    concat_ws("_", col("traffic_type"), col("protocol_type"))
)

In [12]:
df_caracterizado.select(
    "traffic_type",
    "protocol_type",
    "stratum"
).show(10, False)

+------------+-------------+------------+
|traffic_type|protocol_type|stratum     |
+------------+-------------+------------+
|Benign      |Other        |Benign_Other|
|Benign      |TCP          |Benign_TCP  |
|Benign      |Other        |Benign_Other|
|Benign      |Other        |Benign_Other|
|Benign      |TCP          |Benign_TCP  |
|Benign      |Other        |Benign_Other|
|Benign      |TCP          |Benign_TCP  |
|Benign      |TCP          |Benign_TCP  |
|Benign      |TCP          |Benign_TCP  |
|Benign      |TCP          |Benign_TCP  |
+------------+-------------+------------+
only showing top 10 rows


### Construcción de la llave de estratificación

Para aplicar el muestreo estratificado, se construyó una nueva variable llamada **stratum**, la cual combina las variables de caracterización **traffic_type** y **protocol_type**.

Esta llave permite identificar de forma única cada partición Mi dentro de la población. Por ejemplo, un registro clasificado como tráfico benigno y protocolo TCP queda representado como `Benign_TCP`, mientras que un registro malicioso sobre UDP queda representado como `Malicious_UDP`.

La creación de esta variable facilita la aplicación de técnicas de muestreo proporcional por estrato, asegurando que cada grupo identificado en la población original mantenga representación dentro de la muestra M.

La validación de los primeros registros confirma que la llave de estratificación fue generada correctamente y que cada registro conserva su pertenencia a la partición correspondiente.

In [16]:
estratos = [
    row["stratum"]
    for row in df_caracterizado.select("stratum").distinct().collect()
]

fractions = {estrato: 0.05 for estrato in estratos}

muestra_M = df_caracterizado.sampleBy(
    "stratum",
    fractions=fractions,
    seed=42
)

print("Registros en muestra M:", muestra_M.count())
print("Columnas en muestra M:", len(muestra_M.columns))
print(estratos)

Registros en muestra M: 161672
Columnas en muestra M: 90
['Benign_TCP', 'Benign_UDP', 'Benign_Other', 'Malicious_UDP', 'Malicious_TCP', 'Malicious_Other']


### Construcción de la muestra representativa M mediante muestreo estratificado

Una vez identificadas las seis particiones que conforman la población, se procedió a construir la muestra representativa M utilizando un proceso de muestreo estratificado proporcional.

Para ello, se empleó la variable **stratum**, la cual identifica de manera única cada combinación posible entre las variables de caracterización **traffic_type** y **protocol_type**. Los estratos identificados fueron:

- Benign_TCP
- Benign_UDP
- Benign_Other
- Malicious_TCP
- Malicious_UDP
- Malicious_Other

Posteriormente, se aplicó la función `sampleBy()` de PySpark utilizando una fracción de muestreo del 5% para cada estrato. Esta estrategia garantiza que todas las particiones contribuyan proporcionalmente a la construcción de la muestra, preservando la distribución observada en la población original.

El uso de muestreo estratificado resulta especialmente importante debido al fuerte desbalance existente entre las particiones. Mientras que algunas categorías contienen millones de registros, otras cuentan únicamente con algunos miles. Si se hubiera utilizado un muestreo aleatorio simple, los grupos minoritarios podrían haber quedado insuficientemente representados, introduciendo sesgos en las etapas posteriores del análisis.

Como resultado, se obtuvo una muestra M compuesta por **161,672 registros** y **90 variables**, conservando la estructura y diversidad presentes en la población original.

La muestra generada será utilizada en las siguientes etapas del proceso para la construcción de los conjuntos de entrenamiento y prueba, así como para el entrenamiento y evaluación de los modelos de aprendizaje automático.

### Conclusiones de la construcción de la muestra M

La muestra M fue construida mediante una estrategia de muestreo estratificado basada en las variables de caracterización traffic_type y protocol_type. Esta metodología permitió preservar la representatividad de la población original, evitando la pérdida de información asociada a particiones minoritarias y reduciendo el riesgo de introducir sesgos durante el proceso de aprendizaje.

La muestra obtenida conserva la diversidad de comportamientos presentes en el tráfico de red y constituye una base adecuada para las etapas posteriores de entrenamiento, validación y evaluación de modelos de aprendizaje automático.

# 2 Construcción Train – Test

## Paso 2.1 – Construir Train/Test

In [17]:
# Construcción de conjuntos Train-Test

train_df, test_df = muestra_M.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Registros entrenamiento:", train_df.count())
print("Registros prueba:", test_df.count())

Registros entrenamiento: 129580
Registros prueba: 32092


### Construcción de los conjuntos de entrenamiento y prueba

A partir de la muestra representativa M, se construyeron dos subconjuntos independientes: un conjunto de entrenamiento y un conjunto de prueba.

Para esta división se utilizó la función `randomSplit()` de PySpark, aplicando una proporción de **80% para entrenamiento** y **20% para prueba**, junto con una semilla fija `seed=42` para garantizar la reproducibilidad del experimento.

El conjunto de entrenamiento quedó conformado por **129,580 registros**, mientras que el conjunto de prueba quedó conformado por **32,092 registros**.

Esta estrategia permite que el modelo cuente con suficientes observaciones para aprender los patrones presentes en los datos, conservando al mismo tiempo un conjunto independiente para evaluar su capacidad de generalización sobre datos no vistos durante el entrenamiento.

## Paso 2.2 – Calcular porcentajes reales

In [19]:
total = muestra_M.count()

train_pct = (train_df.count() / total) * 100
test_pct = (test_df.count() / total) * 100

print(f"Porcentaje entrenamiento: {train_pct:.2f}%")
print(f"Porcentaje prueba: {test_pct:.2f}%")

Porcentaje entrenamiento: 80.15%
Porcentaje prueba: 19.85%


### Validación de la proporción Train-Test

Una vez realizada la partición de la muestra M, se verificó la proporción real obtenida en cada subconjunto.

Los resultados muestran que el conjunto de entrenamiento representa aproximadamente **80.15%** de la muestra total, mientras que el conjunto de prueba representa **19.85%**. Estas proporciones son consistentes con la estrategia originalmente definida de 80% para entrenamiento y 20% para prueba.

La pequeña diferencia observada respecto a los porcentajes exactos se debe al mecanismo interno de particionamiento aleatorio utilizado por PySpark, el cual distribuye los registros de forma probabilística manteniendo la proporción esperada.

La distribución obtenida es adecuada para el problema de aprendizaje planteado, ya que permite disponer de una cantidad suficiente de observaciones para el entrenamiento de los modelos, conservando simultáneamente un conjunto independiente para evaluar objetivamente el desempeño alcanzado.

## Paso 2.3 – Validar que no haya pérdida de registros

In [20]:
print("Muestra M:", muestra_M.count())
print("Train + Test:", train_df.count() + test_df.count())

Muestra M: 161672
Train + Test: 161672


### Verificación de integridad de la partición

Después de generar los conjuntos de entrenamiento y prueba, se verificó que la totalidad de los registros de la muestra M estuvieran correctamente distribuidos entre ambos subconjuntos.

Los resultados obtenidos muestran que la muestra original contiene **161,672 registros**, mientras que la suma de los registros presentes en los conjuntos de entrenamiento y prueba también es igual a **161,672 registros**.

Esta validación confirma que no se produjo pérdida de información durante el proceso de particionamiento y que cada observación de la muestra fue asignada a uno de los dos subconjuntos generados.

Matemáticamente, se cumple que:

\[
|Train| + |Test| = |M|
\]

lo que garantiza la conservación completa de los datos utilizados en el experimento.

Asimismo, debido a que la partición fue realizada mediante un proceso aleatorio controlado, los conjuntos de entrenamiento y prueba son mutuamente excluyentes, es decir:

\[
Train \cap Test = \emptyset
\]

Esta propiedad es fundamental para evitar fugas de información (*data leakage*) entre las etapas de entrenamiento y evaluación, permitiendo obtener métricas de desempeño confiables y representativas de la capacidad real de generalización de los modelos.

## Paso 2.4 – Verificar representatividad

In [21]:
train_df.groupBy(
    "traffic_type",
    "protocol_type"
).count().orderBy(
    "traffic_type",
    "protocol_type"
).show(truncate=False)

+------------+-------------+------+
|traffic_type|protocol_type|count |
+------------+-------------+------+
|Benign      |Other        |232   |
|Benign      |TCP          |116372|
|Benign      |UDP          |1696  |
|Malicious   |Other        |96    |
|Malicious   |TCP          |10259 |
|Malicious   |UDP          |925   |
+------------+-------------+------+



In [22]:
test_df.groupBy(
    "traffic_type",
    "protocol_type"
).count().orderBy(
    "traffic_type",
    "protocol_type"
).show(truncate=False)

+------------+-------------+-----+
|traffic_type|protocol_type|count|
+------------+-------------+-----+
|Benign      |Other        |47   |
|Benign      |TCP          |28793|
|Benign      |UDP          |404  |
|Malicious   |Other        |24   |
|Malicious   |TCP          |2621 |
|Malicious   |UDP          |203  |
+------------+-------------+-----+



### Validación de representatividad en Train y Test

Finalmente, se verificó la distribución de las particiones en los conjuntos de entrenamiento y prueba, utilizando las variables **traffic_type** y **protocol_type**.

Los resultados muestran que las seis particiones identificadas en la muestra M se conservan tanto en el conjunto de entrenamiento como en el conjunto de prueba:

- Benign_Other
- Benign_TCP
- Benign_UDP
- Malicious_Other
- Malicious_TCP
- Malicious_UDP

Esto confirma que el proceso de división no eliminó ninguna categoría relevante y que ambos subconjuntos mantienen la diversidad de comportamientos presentes en la muestra original.

Aunque las particiones presentan tamaños distintos debido al desbalance natural del dataset, todas se encuentran representadas en ambos subconjuntos. Esto reduce el riesgo de sesgo durante el entrenamiento y permite evaluar el desempeño de los modelos sobre distintos tipos de tráfico y protocolos.

En consecuencia, para cada partición \(M_i\), se cumple que:

\[
M_i = Tr_i \cup Ts_i
\]

y:

\[
Tr_i \cap Ts_i = \emptyset
\]

Por lo tanto, la construcción de los conjuntos Train-Test conserva la estructura de la muestra M y permite realizar una evaluación más confiable de los modelos de aprendizaje automático.

# 3 Selección de métricas para medir calidad de resultados

### Importancia de las métricas de evaluación

Una vez construidos los conjuntos de entrenamiento y prueba, es necesario definir métricas que permitan evaluar objetivamente la calidad de los modelos generados.

La selección de métricas constituye una etapa crítica dentro del proceso de aprendizaje automático, ya que permite cuantificar la capacidad de los modelos para identificar patrones relevantes dentro de los datos y medir su capacidad de generalización sobre observaciones no utilizadas durante el entrenamiento.

Debido a que el problema analizado involucra tráfico de red y detección de comportamientos potencialmente maliciosos, la evaluación no debe basarse únicamente en una sola métrica. En particular, la población presenta un desbalance significativo entre tráfico benigno y tráfico malicioso, por lo que métricas como Accuracy pueden resultar insuficientes para describir completamente el desempeño de un modelo.

Por esta razón, se seleccionaron métricas específicas para los modelos supervisados y no supervisados, considerando tanto la naturaleza de la tarea de aprendizaje como las características de los grandes volúmenes de datos utilizados en este trabajo.

### Métricas seleccionadas para el modelo supervisado

Para evaluar el modelo supervisado Random Forest se utilizarán las siguientes métricas:

#### Accuracy

La métrica Accuracy mide la proporción de predicciones correctas realizadas por el modelo respecto al total de observaciones evaluadas.

\[
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
\]

Esta métrica proporciona una visión general del desempeño del clasificador y permite comparar distintos modelos de manera sencilla.

#### Precision

La métrica Precision mide la proporción de observaciones clasificadas como positivas que realmente pertenecen a la clase positiva.

\[
Precision = \frac{TP}{TP + FP}
\]

En problemas de ciberseguridad esta métrica es importante porque ayuda a reducir la cantidad de falsas alarmas generadas por el sistema.

#### Recall

La métrica Recall mide la capacidad del modelo para identificar correctamente los casos positivos existentes.

\[
Recall = \frac{TP}{TP + FN}
\]

Esta métrica resulta especialmente relevante en tareas de detección de amenazas, donde no identificar un ataque puede representar un riesgo importante para la organización.

#### F1-Score

El F1-Score combina Precision y Recall en una única medida de desempeño.

\[
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
\]

Esta métrica es especialmente útil cuando las clases se encuentran desbalanceadas, como ocurre en el conjunto de datos analizado.

### Métricas seleccionadas para el modelo no supervisado

Para evaluar la calidad del agrupamiento generado por el algoritmo K-Means se utilizarán métricas específicas para aprendizaje no supervisado.

#### Silhouette Score

El coeficiente de Silhouette mide simultáneamente la cohesión interna de cada grupo y la separación existente entre distintos grupos.

Sus valores se encuentran en el intervalo [-1,1].

- Valores cercanos a 1 indican agrupamientos bien definidos.
- Valores cercanos a 0 indican solapamiento entre grupos.
- Valores negativos indican asignaciones incorrectas.

Esta métrica permite evaluar la calidad del agrupamiento sin necesidad de utilizar etiquetas conocidas.

#### Within Set Sum of Squared Errors (WSSSE)

La métrica WSSSE mide la suma de las distancias cuadráticas entre cada observación y el centroide de su cluster correspondiente.

Valores más pequeños indican agrupamientos más compactos y homogéneos.

Esta métrica es ampliamente utilizada para comparar diferentes valores de k y determinar configuraciones adecuadas para el algoritmo K-Means.

### Justificación de la selección de métricas

Las métricas seleccionadas permiten evaluar diferentes aspectos del desempeño de los modelos construidos.

Para el modelo supervisado, Accuracy, Precision, Recall y F1-Score proporcionan una visión integral de la capacidad de clasificación del algoritmo Random Forest, permitiendo identificar tanto la exactitud global como el comportamiento frente a clases minoritarias.

Para el modelo no supervisado, Silhouette Score y WSSSE permiten analizar la calidad estructural de los clusters generados por K-Means, evaluando simultáneamente la separación entre grupos y la cohesión interna de cada agrupamiento.

En conjunto, estas métricas ofrecen una evaluación robusta y adecuada para escenarios Big Data, donde la diversidad, el volumen y el desbalance de los datos pueden influir significativamente en la calidad de los resultados obtenidos.

# 4 Entrenamiento de Modelos de Aprendizaje

## 4.1 Estrategia general de entrenamiento

Una vez construidos los conjuntos de entrenamiento y prueba y definidas las métricas de evaluación, se procedió al entrenamiento de modelos de aprendizaje supervisado y no supervisado.

Para el modelo supervisado se seleccionó el algoritmo Random Forest, debido a su capacidad para manejar grandes volúmenes de datos, modelar relaciones no lineales y reducir el riesgo de sobreajuste mediante el uso de múltiples árboles de decisión.

Para el modelo no supervisado se seleccionó el algoritmo K-Means, ampliamente utilizado para identificar agrupamientos naturales dentro de grandes conjuntos de datos sin requerir etiquetas previamente definidas.

En ambos casos se utilizaron variables numéricas relacionadas con el comportamiento de los flujos de red, seleccionadas por su relevancia para describir características de tráfico legítimo y potencialmente malicioso.

## 4.2 Preparación de datos para Random Forest

In [32]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col, isnan

feature_cols = [
    "Flow Duration",
    "Total Fwd Packet",
    "Total Bwd packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
    "Packet Length Std"
]

rf_data = muestra_M.select(
    *feature_cols,
    col("Label").cast("double").alias("label")
)

print("Registros antes de limpieza:", rf_data.count())

Registros antes de limpieza: 161672


In [33]:
# Limpieza final de datos: eliminación de valores nulos, NaN, Infinity y -Infinity

for c in feature_cols:
    rf_data = rf_data.filter(
        col(c).isNotNull() &
        (~isnan(col(c))) &
        (col(c) != float("inf")) &
        (col(c) != float("-inf"))
    )

rf_data = rf_data.filter(col("label").isNotNull())

print("Registros después de limpieza:", rf_data.count())

Registros después de limpieza: 159568


### Limpieza final de datos para Random Forest

Para el entrenamiento del modelo Random Forest se seleccionaron siete variables numéricas relacionadas con el comportamiento de los flujos de red: duración del flujo, cantidad de paquetes transmitidos, tasa de transferencia y estadísticas del tamaño de los paquetes.

Antes del entrenamiento, se aplicó un proceso de limpieza para eliminar registros con valores nulos, NaN, Infinity y -Infinity. Esta depuración fue necesaria porque los algoritmos de Machine Learning de PySpark requieren que el vector de características contenga únicamente valores numéricos válidos.

Como resultado, el conjunto pasó de **161,672 registros** a **159,568 registros**. Esta reducción representa una pérdida mínima de información y permite garantizar que los datos utilizados para el entrenamiento sean válidos y consistentes.

In [34]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

rf_ready = assembler.transform(rf_data).select(
    "features",
    "label"
)

rf_train, rf_test = rf_ready.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Registros entrenamiento:", rf_train.count())
print("Registros prueba:", rf_test.count())

Registros entrenamiento: 127915
Registros prueba: 31653


### Construcción de features y partición para Random Forest

Después de la limpieza, se utilizó `VectorAssembler` para combinar las variables predictoras en una sola columna llamada **features**, formato requerido por los algoritmos de Machine Learning de PySpark.

Posteriormente, los datos se dividieron en entrenamiento y prueba con una proporción aproximada de 80/20. El conjunto de entrenamiento quedó compuesto por **127,915 registros**, mientras que el conjunto de prueba quedó compuesto por **31,653 registros**.

Esta división permite entrenar el modelo con una cantidad amplia de observaciones y evaluar su desempeño sobre datos no utilizados durante el aprendizaje.

In [35]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=20,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(rf_train)

print("Modelo Random Forest entrenado correctamente")

Modelo Random Forest entrenado correctamente


### Entrenamiento del modelo Random Forest

Se entrenó un modelo Random Forest utilizando la columna **features** como conjunto de variables predictoras y la columna **label** como variable objetivo.

El modelo fue configurado con **20 árboles** y una profundidad máxima de **10 niveles**. Esta configuración permite capturar relaciones complejas en los datos, manteniendo cierto control sobre el crecimiento de los árboles para reducir el riesgo de sobreajuste.

El entrenamiento finalizó correctamente, por lo que el modelo se encuentra listo para generar predicciones sobre el conjunto de prueba y evaluar su desempeño mediante las métricas definidas previamente.

In [36]:
rf_predictions = rf_model.transform(rf_test)

rf_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|prediction|probability                                                                                                                                           |
+-----+----------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|0.0  |0.0       |[0.9903968981276879,0.0070267846531674285,0.002545978420259551,0.0,0.0,0.0,0.0,5.187798298402159E-6,2.5151000586786177E-5,0.0,0.0,0.0]                |
|0.0  |0.0       |[0.9903968981276879,0.0070267846531674285,0.002545978420259551,0.0,0.0,0.0,0.0,5.187798298402159E-6,2.5151000586786177E-5,0.0,0.0,0.0]                |
|0.0  |0.0       |[0.9903968981276879,0.0070267846531674285,0.002545978420259551,0.0,0.0,0.0,0.0,5.187798298402159E-6,2.5151000586786177E-5,0.0,0.0,0.

### Generación de predicciones

Una vez concluido el entrenamiento, el modelo Random Forest fue aplicado al conjunto de prueba para generar predicciones sobre registros no utilizados durante el aprendizaje.

Los resultados muestran la etiqueta real (**label**), la etiqueta predicha (**prediction**) y el vector de probabilidades asociado a cada clase. Estas probabilidades representan el nivel de confianza del modelo para asignar una observación a una categoría específica.

Al observar las primeras predicciones generadas, se aprecia una coincidencia entre las etiquetas reales y las etiquetas predichas, lo que sugiere que el modelo logró capturar patrones relevantes dentro de los datos de tráfico de red.

No obstante, para evaluar objetivamente el desempeño alcanzado, es necesario calcular métricas cuantitativas que permitan medir la calidad de las predicciones realizadas.

In [37]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(rf_predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(rf_predictions)

weighted_precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(rf_predictions)

weighted_recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(rf_predictions)

print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Precision:", weighted_precision)
print("Recall:", weighted_recall)

Accuracy: 0.9722617129497994
F1 Score: 0.972112633428322
Precision: 0.9720348283017639
Recall: 0.9722617129497995


### Evaluación del modelo Random Forest

Para evaluar el desempeño del modelo supervisado se calcularon las métricas definidas previamente: Accuracy, Precision, Recall y F1 Score.

Los resultados obtenidos sobre el conjunto de prueba fueron los siguientes:

| Métrica | Valor |
|----------|----------|
| Accuracy | 0.9723 |
| Precision | 0.9720 |
| Recall | 0.9723 |
| F1 Score | 0.9721 |

El modelo alcanzó una exactitud superior al 97%, lo que indica una elevada capacidad para clasificar correctamente los distintos tipos de tráfico presentes en el conjunto de datos.

Asimismo, los valores obtenidos para Precision y Recall son muy similares entre sí, lo que sugiere un comportamiento equilibrado entre la capacidad para identificar correctamente los casos positivos y la capacidad para evitar clasificaciones incorrectas.

Por su parte, el F1 Score presenta un valor cercano al Accuracy, confirmando que el modelo mantiene un desempeño consistente incluso considerando el desbalance existente entre las clases presentes en el dataset.

En conjunto, estos resultados indican que el algoritmo Random Forest fue capaz de aprender patrones relevantes dentro de los datos de tráfico de red y constituye una alternativa efectiva para tareas de clasificación en escenarios de ciberseguridad.

## 4.3 Entrenamiento del modelo no supervisado K-Means

In [38]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=3,
    seed=42
)

kmeans_model = kmeans.fit(rf_ready)

print("Modelo K-Means entrenado correctamente")

Modelo K-Means entrenado correctamente


### Entrenamiento del modelo K-Means

Como complemento al modelo supervisado, se implementó un modelo de aprendizaje no supervisado utilizando el algoritmo K-Means.

Este algoritmo busca identificar agrupamientos naturales dentro de los datos sin utilizar la variable objetivo. Para este experimento se utilizaron las mismas variables numéricas empleadas en Random Forest, permitiendo comparar ambos enfoques de aprendizaje sobre un conjunto de características consistente.

El modelo fue configurado con **k = 3 clusters**, con el objetivo de identificar diferentes perfiles de comportamiento presentes en el tráfico de red analizado.

Una vez finalizado el proceso de entrenamiento, el modelo quedó preparado para asignar cada observación a un grupo específico y evaluar la calidad del agrupamiento mediante métricas de cohesión y separación entre clusters.

In [42]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(rf_ready)

scaled_data = scaler_model.transform(rf_ready)

scaled_data.select("scaled_features").show(5, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------+
|scaled_features                                                                                                                                 |
+------------------------------------------------------------------------------------------------------------------------------------------------+
|[3.4179872961658546,-0.18293835002861403,-0.1736631446732829,-0.08144100012395745,-0.3874653435846651,-0.34775381006142775,-0.39759777117104994]|
|[3.418002771814348,-0.18293835002861403,-0.1736631446732829,-0.08144100012395745,-0.38746534362558366,-0.34775381006142775,-0.39759777117104994]|
|[-0.28647467739553273,-0.18004803857273935,-0.17197114588705226,0.4297772974178397,1.122829174283816,0.5687992590114035,0.7822831466173416]     |
|[-0.28311204005862217,-0.17571257138892735,-0.1652031507421297,-0.07098444397549387,-0.3660923151793351,0.97640268398

In [43]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

kmeans_scaled = KMeans(
    featuresCol="scaled_features",
    predictionCol="cluster",
    k=3,
    seed=42
)

kmeans_scaled_model = kmeans_scaled.fit(scaled_data)

kmeans_scaled_predictions = kmeans_scaled_model.transform(scaled_data)

evaluator_scaled = ClusteringEvaluator(
    predictionCol="cluster",
    featuresCol="scaled_features",
    metricName="silhouette"
)

silhouette_scaled = evaluator_scaled.evaluate(kmeans_scaled_predictions)

print("Silhouette Score con escalamiento:", silhouette_scaled)

Silhouette Score con escalamiento: 0.7727470142141075


### Evaluación del modelo K-Means

Inicialmente se realizó una prueba de agrupamiento utilizando las variables originales sin escalamiento. Sin embargo, debido a las diferencias significativas de magnitud entre las características seleccionadas, los resultados obtenidos mostraron una baja calidad de agrupamiento.

Por esta razón, se aplicó un proceso de estandarización mediante la técnica StandardScaler, transformando todas las variables a una escala comparable antes de ejecutar nuevamente el algoritmo K-Means.

Posteriormente se entrenó el modelo utilizando tres clusters (k = 3) y se evaluó la calidad del agrupamiento mediante el coeficiente de Silhouette.

El resultado obtenido fue:

| Métrica | Valor |
|----------|----------|
| Silhouette Score | 0.7727 |

Este valor indica una adecuada separación entre los grupos identificados y una buena cohesión interna dentro de cada cluster. Aunque no representa una separación perfecta, el resultado puede considerarse satisfactorio para un problema de análisis de tráfico de red con múltiples patrones de comportamiento.

Los resultados demuestran que el escalamiento de variables constituye una etapa fundamental cuando se utilizan algoritmos basados en distancia, como K-Means, especialmente en escenarios Big Data donde las variables pueden presentar órdenes de magnitud muy diferentes.

# 5 Análisis de resultados

Los resultados obtenidos muestran que el proceso desarrollado permitió construir y evaluar modelos de aprendizaje automático adecuados para el análisis de grandes volúmenes de datos de tráfico de red. A lo largo de la actividad se cuidó que la muestra M conservara la representatividad de la población original, utilizando variables de caracterización como `traffic_type` y `protocol_type`. Esto fue importante porque el dataset presentaba un desbalance evidente entre tráfico benigno, tráfico malicioso y diferentes protocolos.

En el caso del modelo supervisado Random Forest, los resultados fueron altamente satisfactorios. El modelo obtuvo un Accuracy de **0.9723**, Precision de **0.9720**, Recall de **0.9723** y F1 Score de **0.9721**. Estos valores indican que el modelo fue capaz de clasificar correctamente la mayoría de los registros del conjunto de prueba. Además, la similitud entre Precision, Recall y F1 Score sugiere que el desempeño no depende únicamente de la clase mayoritaria, sino que existe un equilibrio razonable entre la detección correcta y la reducción de errores de clasificación.

Desde una perspectiva práctica, este resultado es relevante para escenarios de ciberseguridad, ya que un modelo con buen Recall puede ayudar a reducir la probabilidad de dejar pasar tráfico malicioso sin detectar, mientras que una buena Precision ayuda a disminuir falsas alarmas. En un entorno real, ambos aspectos son importantes: detectar amenazas es fundamental, pero generar demasiadas alertas incorrectas también puede afectar la operación de los equipos de seguridad.

Por otro lado, el modelo no supervisado K-Means permitió explorar patrones internos dentro de los datos sin utilizar la variable objetivo. Durante el proceso se observó que K-Means es muy sensible a la escala de las variables. Al utilizar las características originales, algunas variables como `Flow Duration` dominaban el cálculo de distancias, afectando negativamente la calidad del agrupamiento. Después de aplicar `StandardScaler`, el Silhouette Score alcanzó un valor de **0.7727**, lo cual representa una mejora importante y evidencia una separación adecuada entre clusters.

Este resultado confirma que el preprocesamiento no es una etapa secundaria, sino una parte esencial del modelado. En particular, para algoritmos basados en distancia como K-Means, escalar las variables puede marcar una diferencia significativa entre obtener agrupamientos poco útiles o identificar estructuras realmente interpretables dentro de los datos.

Como fortaleza principal del trabajo, se logró construir una metodología completa: selección de muestra representativa, particionamiento Train-Test, limpieza de datos, entrenamiento de modelos y evaluación mediante métricas apropiadas. Además, se utilizaron enfoques supervisados y no supervisados, lo que permitió analizar el problema desde dos perspectivas distintas: clasificación con etiquetas conocidas y descubrimiento de patrones sin etiquetas.

Como área de oportunidad, sería conveniente ampliar la experimentación con más algoritmos y configuraciones de hiperparámetros. Por ejemplo, se podrían probar Gradient-Boosted Trees, Logistic Regression o diferentes valores de k para K-Means. También sería recomendable analizar con mayor detalle la matriz de confusión para identificar qué clases presentan mayor dificultad de clasificación.

En conclusión, Random Forest mostró el mejor desempeño para la tarea supervisada de clasificación, mientras que K-Means resultó útil como herramienta exploratoria para identificar grupos de comportamiento en el tráfico de red. Los resultados obtenidos permiten concluir que las métricas seleccionadas fueron adecuadas para evaluar la calidad de los modelos y que PySpark representa una herramienta apropiada para procesar y analizar grandes volúmenes de datos en contextos de ciberseguridad.